# ☁️ SubtitleStudioX – Google Colab

Ejecuta las celdas en orden:
- **Celda 1**: Instala el entorno (3-5 min)
- **Celda 2**: Procesa videos en bucle continuo

> 💻 ¿Prefieres tu PC? Usa [SubtitleStudio](https://github.com/KingEdhard/SubtitleStudio)

In [ ]:
# ============================================
# CELDA 1: Instalación del entorno
# ============================================
import os
import subprocess

usuario = "KingEdhard"
proyecto = "SubtitleStudioX"
url_git = f"https://github.com/{usuario}/{proyecto}.git"

print(f"📦 Clonando proyecto desde: {url_git}")
with open('instalador_seguro.sh', 'w') as f:
    f.write(f"#!/bin/bash\nrm -rf {proyecto} github.com\ngit clone {url_git}\ncd {proyecto}\npip install -r requirements.txt --quiet\n")

!chmod +x instalador_seguro.sh && ./instalador_seguro.sh

print("📦 Instalando motor WhisperX...")
!pip install "git+https://github.com/m-bain/whisperx.git" --quiet

print("⚙️ Configurando FFmpeg...")
!apt-get install ffmpeg -y > /dev/null
%cd /content/{proyecto}
!mkdir -p bin
!ln -sf $(which ffmpeg) bin/ffmpeg.exe
!ln -sf $(which ffprobe) bin/ffprobe.exe

print("\n✅ Entorno listo. Ejecuta la Celda 2.")

In [ ]:
# ============================================
# CELDA 2: Procesamiento en bucle continuo
# ============================================
import os
import sys
from google.colab import files

%cd /content/SubtitleStudioX

print("⚙️ Instalando PyTorch compatible con WhisperX (toma 1-2 min)...")
!pip uninstall torch torchvision torchaudio -y --quiet 2>/dev/null
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --quiet --force-reinstall 2>/dev/null
print("✅ PyTorch estabilizado.")

print("🔍 Verificando WhisperX...")
try:
    import whisperx
    print("✅ WhisperX detectado.")
except ImportError:
    !pip install "git+https://github.com/m-bain/whisperx.git" --quiet
    print("✅ WhisperX instalado.")

print("\n[⚙️] Generando main.py optimizado...")

codigo_definitivo = '''import sys
import os
import torch
from tqdm import tqdm
from src.components.extraccion import extraer_audio_mejorado
from src.components.traduccion import traducir_srt
from src.components.muxer import incrustar_subtitulos

def transcribir_con_whisperx(wav_path):
    import whisperx
    device, compute_type = "cuda", "float16"
    
    print("\\n🚀 [IA] Iniciando WhisperX con el mejor modelo (Large-v3)...")
    model = whisperx.load_model("large-v3", device, compute_type=compute_type, language="en")
    audio = whisperx.load_audio(wav_path)
    result = model.transcribe(audio, batch_size=16)
    
    print("🎯 [IA] Aplicando alineación fonética milimétrica (Wav2Vec2)...")
    model_a, metadata = whisperx.load_align_model(language_code="en", device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
    
    srt_path = wav_path.replace("_dialogos_mejorados.wav", "_en.srt")
    
    def format_time(t):
        h, m, s, ms = int(t // 3600), int((t % 3600) // 60), int(t % 60), int((t % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
        
    with open(srt_path, "w", encoding="utf-8") as f:
        for idx, segment in enumerate(result["segments"], 1):
            start = format_time(segment["start"])
            end = format_time(segment["end"])
            f.write(f"{idx}\\n{start} --> {end}\\n{segment['text'].strip()}\\n\\n")
            
    print(f"✔ Subtítulos milimétricos generados con éxito: {srt_path}")
    return srt_path

def main():
    print("\\n=== SUBTITLESTUDIOX + WHISPERX (Versión Ultra Precisa) ===")
    print("Tu ingeniería de audio + Alineación milimétrica + Tu traducción latino\\n")
    
    formatos_video = ('.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv')
    archivos = [f for f in os.listdir('.') if f.lower().endswith(formatos_video)]
    if not archivos:
        print("❌ No se encontró ningún archivo de vídeo en la carpeta actual.")
        return
        
    total = len(archivos)
    print(f"📂 {total} vídeo(s) detectado(s).")
    for i, video in enumerate(archivos, 1):
        print(f"\\n{'='*60}\\n🎬 Procesando: {video}\\n{'='*60}")
        try:
            wav = extraer_audio_mejorado(video)
            if not wav: continue
            srt_ingles = transcribir_con_whisperx(wav)
            if not srt_ingles:
                if os.path.exists(wav): os.remove(wav)
                continue
            print("\\n🌎 Traduciendo subtítulos a español latino con tu modelo...")
            srt_espanol = traducir_srt(srt_ingles)
            ruta_final = incrustar_subtitulos(video, srt_ingles, srt_espanol if srt_espanol else srt_ingles)
            if ruta_final:
                print(f"\\n🏁 ¡Vídeo completado con éxito!: {ruta_final}")
            if os.path.exists(wav): os.remove(wav)
        except Exception as e:
            print(f"\\n❌ Error procesando {video}: {e}")
            if 'wav' in locals() and os.path.exists(wav): os.remove(wav)
            continue

if __name__ == '__main__':
    main()'''

with open("main.py", "w", encoding="utf-8") as f:
    f.write(codigo_definitivo)

print("✅ main.py generado.")
print("-" * 60)

# Bucle infinito de carga y procesamiento
while True:
    formatos_video = ('.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv')
    videos = [f for f in os.listdir('.') if f.lower().endswith(formatos_video)]
    
    if videos:
        print(f"\n📂 {len(videos)} video(s) detectado(s). Procesando...\n")
    else:
        print("\n🎬 Sube tus videos (o cancela para salir):")
        uploaded = files.upload()
        if not uploaded:
            resp = input("\n¿Salir? (S/n): ")
            if resp.upper() == 'S':
                break
            else:
                continue
    
    !python main.py
    print("\n✅ Lote completado.")
    
    resp = input("\n🔄 ¿Cargar más videos? (Enter=Sí | S=Salir): ")
    if resp.upper() == 'S':
        break
    for v in videos:
        try:
            os.remove(v)
        except:
            pass

print("\n🎉 ¡Sesión finalizada! Descarga tus videos del panel izquierdo 📁")